# Day 04：核心数据结构（上）（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-004` ｜ **开发分支**：`feat/day04-list-tuple-set-todo`  
> **配套 HTML5 课件**：`courseware/day04/index.html` ｜ **独立版本快照**：`versions/day04/`

---

## 🎙️ 老陈开场旁白

今天咱们正式进入 Python 最核心的三大序列/集合容器：列表（list）、元组（tuple）、集合（set）。别小看这仨基础容器，你在 AI 工程里天天跟它们打交道：多轮对话历史 `messages` 是 list，不可变的模型配置快照是 tuple，RAG 召回文档去重靠的是 set！今天 STARMIND-004 林产品要一个小工具：研发团队用的「命令行待办事项管理器（Todo CLI）」，支持新增、完成、筛选、删除任务，同时附带一个「知识库标签集合分析器」，帮运营找出两批文档的共同标签和新增标签。

### 承上启下
- **昨日基线**：昨天咱们用流程控制做出了工单三级分流器和命令行菜单，但数据还是零散的单个变量，没法动态增删改查。
- **明日预告**：明天（Day 5）是第一周的重头戏——字典（dict）与 JSON 深度实战！所有大模型 API 的请求和响应全都是嵌套 JSON，明天彻底打通。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day04'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-004）

星辰智脑项目组进入密集迭代期，团队需要轻量级任务清单工具追踪每日待办，同时知识库导入文档时存在大量重复标签需要自动去重与比对。

**今日迭代目标：**
- 掌握 list 的增删改查、排序、列表推导式与浅拷贝安全遍历。
- 理解 tuple 的不可变特性及多值解包（unpacking）用法。
- 熟练运用 set 的去重及交集（&）、并集（|）、差集（-）运算。

**验收标准：**
- 待办管理器支持添加任务、按编号标记完成、删除任务、按状态过滤及列表统计。
- 标签分析器能准确计算两批文档标签的去重全集、公共标签、新增与移除标签。
- pytest 单元测试全部通过，包含空列表与越界索引边界处理。

---

## 📚 课堂核心技术笔记

### 1. 1. list 列表与列表推导式

`list` 是动态数组，支持 `append`、`extend`、`pop`、`sort` 与切片。列表推导式 `[x.strip() for x in raw if x.strip()]` 不仅比手写循环更简洁，在 CPython 底层执行速度也更快。

> **⚠️ 老陈避坑**：致命坑：永远不要在 `for item in my_list:` 循环中调用 `my_list.remove(item)`！删除元素会导致后续元素前移，直接跳过下一个元素。正确做法是用列表推导式新建过滤后的列表。

### 2. 2. tuple 元组与解包

`tuple` 一旦创建不可增删改元素，占用内存比 `list` 更小，还能作为字典的 key 或放进 `set` 里。单元素元组必须加逗号：`(42,)`，否则 `(42)` 只是普通整数！支持优雅解包：`task_id, title, done = task`。

> **⚠️ 老陈避坑**：如果元组里嵌套了一个列表 `t = (1, [2, 3])`，虽然不能替换 `t[1]` 的引用，但里面的列表本身依然可以被 `t[1].append(4)` 修改。

### 3. 3. set 集合与哈希去重

`set` 基于哈希表实现，自动去重且成员检测 `x in s` 为 O(1) 常数时间。创建空集合必须用 `set()`，因为 `{}` 创建的是空字典！

> **⚠️ 老陈避坑**：`set` 是无序的，如果你需要「既去重又保持原始顺序」，可以用 `list(dict.fromkeys(items))`（Python 3.7+ 字典保序特性）。


## 💻 实操步骤 1：步骤一：实现待办事项管理器核心引擎（list + tuple）

- **目标文件**：`versions/day04/src/day04_todo_manager.py`
- **老陈代码旁白**：咱们用 `list` 存所有待办，每条任务用不可变元组 `(task_id, title, is_done)` 表达。看看怎么优雅实现增删改查和统计。


In [ ]:
"""
STARMIND-004 模块一：星辰智脑研发待办事项管理器（Todo CLI 引擎）。
演示 list 增删改查、tuple 不可变记录与解包、列表推导式过滤。
"""

from typing import List, Tuple

# 定义类型别名：单条任务为 (任务ID, 任务标题, 是否完成)
TaskRecord = Tuple[int, str, bool]


def add_task(tasks: List[TaskRecord], title: str) -> TaskRecord:
    """向任务列表末尾添加一条新待办，并返回新建的任务元组。"""
    cleaned = title.strip()
    if not cleaned:
        raise ValueError("任务标题不能为空")
    # 计算下一个自增 ID（即使中间删除过任务也不会冲突）
    next_id = max((t[0] for t in tasks), default=0) + 1
    record: TaskRecord = (next_id, cleaned, False)
    tasks.append(record)
    return record


def complete_task(tasks: List[TaskRecord], task_id: int) -> bool:
    """按任务 ID 将任务标记为已完成，成功返回 True，未找到返回 False。"""
    for idx, (tid, title, _done) in enumerate(tasks):
        if tid == task_id:
            # 元组不可变，因此构造新元组替换原位置元素
            tasks[idx] = (tid, title, True)
            return True
    return False


def remove_task(tasks: List[TaskRecord], task_id: int) -> bool:
    """安全删除指定 ID 的任务（避免遍历时原地 remove 引发错位）。"""
    original_len = len(tasks)
    # 使用切片赋值 tasks[:] 原地替换为推导式过滤后的新列表
    tasks[:] = [t for t in tasks if t[0] != task_id]
    return len(tasks) < original_len


def filter_tasks(tasks: List[TaskRecord], only_pending: bool = False) -> List[TaskRecord]:
    """根据完成状态筛选任务列表。"""
    if only_pending:
        return [t for t in tasks if not t[2]]
    return list(tasks)


def summarize_tasks(tasks: List[TaskRecord]) -> Tuple[int, int, int]:
    """返回只读统计快照：(总任务数, 已完成数, 待办数)。"""
    total = len(tasks)
    done_count = sum(1 for _, _, done in tasks if done)
    return (total, done_count, total - done_count)


if __name__ == "__main__":
    todo_list: List[TaskRecord] = []
    add_task(todo_list, "配置星辰智脑开发环境")
    add_task(todo_list, "完成客服文本清洗器")
    add_task(todo_list, "编写 Day 4 单元测试")
    complete_task(todo_list, 1)
    remove_task(todo_list, 2)

    total, done, pending = summarize_tasks(todo_list)
    print(f"任务概览 -> 总计:{total} | 已完成:{done} | 待办:{pending}")
    for tid, title, is_done in todo_list:
        mark = "[x]" if is_done else "[ ]"
        print(f"  {mark} #{tid} {title}")


### 🔍 步骤 1 代码深度解析

1. `tasks[:] = [...]` 是切片原地替换技巧：既保留了外部传入列表对象的内存引用，又安全完成了批量过滤删除。
2. `max((t[0] for t in tasks), default=0) + 1` 确保哪怕删掉第 2 条任务再新增，ID 依然严格递增不会撞号。

**预期运行输出参考：**
```text
任务概览 -> 总计:2 | 已完成:1 | 待办:1
  [x] #1 配置星辰智脑开发环境
  [ ] #3 编写 Day 4 单元测试
```


## 💻 实操步骤 2：步骤二：实现知识库标签去重与差异分析器（set）

- **目标文件**：`versions/day04/src/day04_tag_analyzer.py`
- **老陈代码旁白**：林产品还希望对比两版知识库文档的标签差异，看哪些是新增主题、哪些下线了。用 `set` 写出来干净得像数学公式。


In [ ]:
"""
STARMIND-004 模块二：知识库文档标签去重与集合差异分析器。
演示 set 创建、去重清洗、交集(&)/并集(|)/差集(-)与排序输出。
"""

from typing import List, Dict, Set


def normalize_tags(raw_tags: List[str]) -> Set[str]:
    """清洗并去重标签：去首尾空格、转小写、过滤空串。"""
    # 使用集合推导式一步完成清洗与去重
    return {tag.strip().lower() for tag in raw_tags if tag and tag.strip()}


def compare_tag_versions(old_tags: List[str], new_tags: List[str]) -> Dict[str, List[str]]:
    """
    对比旧版与新版知识库标签，返回确定顺序（已排序）的差异报告：
    - common: 两版共同保留的标签
    - added: 新版新增的标签
    - removed: 新版移除的标签
    - all_unique: 两版合并后的全部唯一标签
    """
    set_old = normalize_tags(old_tags)
    set_new = normalize_tags(new_tags)

    return {
        "common": sorted(set_old & set_new),
        "added": sorted(set_new - set_old),
        "removed": sorted(set_old - set_new),
        "all_unique": sorted(set_old | set_new),
    }


if __name__ == "__main__":
    v1_tags = [" Python ", "Prompt", "git", "python", ""]
    v2_tags = ["python", "RAG", "Agent", "Prompt ", "RAG"]
    diff = compare_tag_versions(v1_tags, v2_tags)
    for k, v in diff.items():
        print(f"{k:12s}: {v}")


### 🔍 步骤 2 代码深度解析

1. `{tag.strip().lower() for tag in raw_tags if tag and tag.strip()}` 是集合推导式，既洗掉了大小写不一致的 `'Python'` 和 `'python'`，又自动去重。
2. 输出前套一层 `sorted(...)`，因为 `set` 本身无序，排序转成 `list` 才能保证接口响应与单元测试断言结果每次完全一致。

**预期运行输出参考：**
```text
common      : ['prompt', 'python']
added       : ['agent', 'rag']
removed     : ['git']
all_unique  : ['agent', 'git', 'prompt', 'python', 'rag']
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day04/tests/test_day04.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-004 自动化单元测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day04_todo_manager import add_task, complete_task, remove_task, filter_tasks, summarize_tasks
from day04_tag_analyzer import normalize_tags, compare_tag_versions


def test_todo_crud_lifecycle():
    tasks = []
    t1 = add_task(tasks, "任务A")
    t2 = add_task(tasks, "任务B")
    assert t1 == (1, "任务A", False)
    assert t2 == (2, "任务B", False)

    assert complete_task(tasks, 1) is True
    assert complete_task(tasks, 999) is False
    assert summarize_tasks(tasks) == (2, 1, 1)
    assert len(filter_tasks(tasks, only_pending=True)) == 1

    assert remove_task(tasks, 1) is True
    assert remove_task(tasks, 1) is False
    assert summarize_tasks(tasks) == (1, 0, 1)

    with pytest.raises(ValueError):
        add_task(tasks, "   ")


def test_tag_analyzer():
    assert normalize_tags([" AI ", "ai", "", "LLM"]) == {"ai", "llm"}
    diff = compare_tag_versions(["Python", "Git"], ["python", "RAG"])
    assert diff["common"] == ["python"]
    assert diff["added"] == ["rag"]
    assert diff["removed"] == ["git"]
    assert diff["all_unique"] == ["git", "python", "rag"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：保序去重工具

**题目要求：**
RAG 召回的文档 ID 列表如 `['doc3', 'doc1', 'doc3', 'doc2', 'doc1']`，要求去除重复 ID 但必须严格保持它们首次出现的先后顺序（因为顺序代表相关性排名）。编写 `dedupe_keep_order(items: list) -> list`。

**思路提示：** 方法一：用辅助 `set` 记录已见元素；方法二：一行 `list(dict.fromkeys(items))`。


In [ ]:
"""Day 4 课后题 1：RAG 召回文档保序去重。"""

from typing import List


def dedupe_keep_order(items: List[str]) -> List[str]:
    """去除重复元素并严格保留首次出现的顺序。"""
    seen = set()
    result: List[str] = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


if __name__ == "__main__":
    recalled = ["doc3", "doc1", "doc3", "doc2", "doc1"]
    print("保序去重结果:", dedupe_keep_order(recalled))


**答案解析与评分标准：**

经典高频面试题！用 `seen` 集合将判重降为 O(1)，整体时间复杂度 O(n)，比用 `if item not in result` 的 O(n²) 快得多。


### 作业 2（企业实战）：企业实战：多模型权限交集校验器

**题目要求：**
员工拥有若干权限标签（如 `{'chat', 'rag', 'code'}`），某个 AI 工具需要一组必备权限（如 `{'rag', 'export'}`）。编写 `check_permissions(user_perms: set, required_perms: set) -> tuple[bool, list[str]]`，返回是否放行以及缺失的权限列表（已排序）。

**思路提示：** 缺失权限就是集合差集 `required_perms - user_perms`。


In [ ]:
"""Day 4 课后题 2：AI 工具权限集合校验器。"""

from typing import Set, Tuple, List


def check_permissions(user_perms: Set[str], required_perms: Set[str]) -> Tuple[bool, List[str]]:
    """检查用户是否具备全部所需权限，返回 (是否通过, 缺失权限列表)。"""
    missing = sorted(required_perms - user_perms)
    return (len(missing) == 0, missing)


if __name__ == "__main__":
    print(check_permissions({"chat", "rag"}, {"rag", "export"}))
    print(check_permissions({"chat", "rag", "export"}, {"rag", "export"}))


**答案解析与评分标准：**

利用集合差集一行算出缺失权限，是企业 RBAC 权限系统中最常见的底层实现。


### 作业 3（架构进阶）：架构进阶：列表推导式实现滑动窗口切块

**题目要求：**
后续做 RAG 文档切分时，常需要按固定窗口大小 `size` 和步长 `step` 对文本列表做滑动切片。编写 `sliding_window(items: list, size: int, step: int) -> list[tuple]`，返回所有完整窗口的元组列表。

**思路提示：** 用 `range(0, len(items) - size + 1, step)` 生成起始索引，配合切片 `tuple(items[i:i+size])`。


In [ ]:
"""Day 4 课后题 3：滑动窗口切块器（RAG 分块原型）。"""

from typing import List, Tuple, Any


def sliding_window(items: List[Any], size: int, step: int = 1) -> List[Tuple[Any, ...]]:
    """按窗口大小与步长切分序列，返回不可变窗口元组列表。"""
    if size <= 0 or step <= 0 or size > len(items):
        return []
    return [tuple(items[i : i + size]) for i in range(0, len(items) - size + 1, step)]


if __name__ == "__main__":
    sentences = ["句1", "句2", "句3", "句4", "句5"]
    print("窗口=3, 步长=2:", sliding_window(sentences, size=3, step=2))


**答案解析与评分标准：**

这道题直接铺垫了第三阶段 RAG 的 Chunk Overlap（重叠分块）算法，用列表推导式 + 切片即可优雅实现。


---

## 🎯 老陈收尾复盘

记住今天容器选型的黄金口诀：有序且要增删改用 `list`，只读防篡改用 `tuple`，快速判重和交并差集无脑用 `set`！特别提醒一句：在百万级数据里判断 `x in container`，`list` 是 O(n) 慢如蜗牛，`set` 是 O(1) 哈希直达。明天咱们攻克字典与 JSON，做好准备！
